In [1]:
from datetime import datetime
import os
from typing import Literal

import pandas as pd
from tinydb import TinyDB, Query

from outlines import models, generate
from openai import AsyncOpenAI
from outlines.models.openai import OpenAIConfig

from minio import Minio
from dotenv import dotenv_values

In [2]:
config_dict = dotenv_values(".env")

db = TinyDB("retrieval_results/db.json")

In [3]:
# Use the best encoder, metric, and method from retrieval comparison results
ENCODER = "Alibaba-NLP/gte-Qwen2-7B-instruct"
METRIC = "dot"
METHOD = "12"
Item = Query()

test_dict = db.search(
    (Item.encoder == ENCODER) & 
    (Item.distance == METRIC) & 
    (Item.method == METHOD))

In [4]:
prompt_template = """You are an expert data curator. You are given a product name and the commercial category where it belongs.
Your task is to find the most similar match from a list of possible options. 
If no option is suitable, you should output "none of the above". The options are:

1. {options[0]}
2. {options[1]}
3. {options[2]}
4. {options[3]}
5. {options[4]}
6. {options[5]}
7. {options[6]}
8. {options[7]}
9. {options[8]}
10. {options[9]}
11. {options[10]}
12. {options[11]}
13. none of the above

Your output should only be the exact text of one of the options above, and nothing else.

The product name is: {name}
The commercial category where the product belongs is: {category}
The most similar option is: """

In [12]:
modelname = "deepseek-r1:70b"
#modelname = "deepseek-r1:8b"
#modelname = "qwen2.5:32b"
#modelname = "llama3.3:70b"

prod=test_dict[0]

In [13]:
# initialize model
llmclient = AsyncOpenAI(
    api_key="none",
    base_url='http://localhost:11434/v1/',
)
config = OpenAIConfig(model=modelname, temperature=0)
model = models.openai(llmclient, config)
safe_modelname = modelname.replace(".", "").replace(":", "-")

In [14]:
search_dict = prod.get("search_dict")
options = list(search_dict.keys()) + ["none of the above"]
generator = generate.choice(model, options)
#pattern = "|".join(list(search_dict.keys()) + ["none of the above"])
#generator = generate.regex(model, pattern)

In [15]:
res = generator(prompt_template.format(options=options, name=prod["item"]["name"], category=prod["item"]["category"]))

In [16]:
res

'none of the above'